In [1]:
import itertools
import logging
import pathlib

import altair as alt
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.optimize import minimize
from scipy.stats import norm

import pitchmark.osm

alt.renderers.enable('mimetype')
logger = logging.getLogger(__name__)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Adapted from Penner (Can J Phys 2002): The physics of putting

## Level green (Stimpmeter)

A golf ball with mass $m$ and moment of inertia $I$ is rolling along the $y$-direction on a level green ($x=z=0$).
The friction force (of the deformed green on the ball) is resolved in a component $n$ perpendicular to the surface and a component $f$ parallel to the surface.
The golf ball has radius $R$ and the friction force acts a distance $\rho$ ahead of and $R_t=\sqrt{R^2-\rho^2} \approx R$ below the center of mass.

The equations of motion are:

$$
ma_y = -f \\
I\alpha_x = n\rho - fR_t
$$

To simplify the analysis considerably, the ball is assumed to be rolling throughout, i.e.

$$
a_y = -\alpha_x R
$$

No acceleration in the $z$-direction means

$$
n = mg
$$

and solving all the above gives

$$
a_y = \frac{-g}{1 + \frac{I}{m R^2}} \frac{\rho}{R}
$$

which Penner gives as 

$$
a_y = -\frac{5}{7} \rho_g g
$$

where $\rho_g = \frac{\rho}{R}$ and using $I = I_b m R^2 = \frac{2}{5}mR^2$ for a uniform sphere.

A golf ball leaves a Stimpmeter at a speed of $v_0 = 6 $ (in ft/s) and travels a distance $y$ (also in feet).
It will come to a complete stop after a time 

$$
t = -\frac{v_0}{a_y}
$$

and the distance travelled obeys

$$
y = v_o t + \frac{1}{2} a_y t^2
$$

such that 

$$
y = - \frac{v_0^2}{2 a_y}
$$

Using $a_y$ from above gives an expression for $\rho_g$ in terms of the Stimpmeter reading $y$:

$$
\rho_g = \frac{v_0^2}{2 y} \frac{1 + \frac{I}{mR^2}}{g} \\
\rho_g = \frac{(6 ft/s)^2}{2 y} \frac{1 + I_b}{32.1 ft/s^2} \\
\rho_g = \frac{126.0 ft}{5 \times 32.1 y} \\
\rho_g = \frac{0.785 ft}{y}
$$

In [ ]:
def softness(stimp_reading, *, ball_moi = 0.4):
    init_ball_speed = 6  # in feet per second
    gravity = 32.1  # feet / second^2
    return (init_ball_speed**2) * (1.0 + ball_moi) / (2 * gravity * stimp_reading)

In [ ]:
softness(np.linspace(4.0, 13.0, 10))

In [ ]:
np.sqrt(1.0 - softness(np.linspace(4.0, 13.0, 10))**2)

Consider now a sloped green with angles with respect to the horizontal of $\theta$ along the $x$-axis, and of $\psi$ along the $y$-axis.
The gravitational force (weight) is given as
$$
\vec{W} = -mg (\sin \theta, \cos \theta \sin \psi, \cos \theta \cos \psi).
$$
by Penner, but I do not believe this because it is not symmetrical if $x$ and $y$ (and $\theta$ and $\psi$) are interchanged.

In the case where $\psi = 0$, $\bm{\hat{\jmath}} = \bm{\hat{y}}$ and the weight vector should be given by
$$
\vec{W} = -mg(\sin \theta, 0, \cos \theta)
$$
and where $\theta = 0$, it is
$$
\vec{W} = -mg
$$


The ball is assumed to be moving with velocity $\vec{v}$ which makes an angle $\beta$ to the positive $y$-axis, and the friction force is given by $\vec{f}$ which makes an angle $\phi$ with the negative $y$-axis. (See Penner Fig. 2)
The friction force is
$$
\vec{f} = (-f \sin \phi, -f \cos \phi, n).
$$
(Should also be the other way around?)

In [ ]:
GRAVITY = 9.8  # m/s^2
MOI_SOLID_SPHERE = 0.4
BALL_RADIUS = 0.02135  # m
HOLE_RADIUS = 0.054  # m

In [ ]:
softness(12)

In [ ]:
class Green:
    def __init__(
        self,
        stimp,
        *,
        gdf=None,
        hole_location=None,
        hole_radius=HOLE_RADIUS,
        holing_vmax=1.63
    ):
        self.stimp = stimp
        self.softness = softness(stimp, ball_moi=MOI_SOLID_SPHERE)
        self.gdf = gdf
        self.hole_location = (0.0, 0.0) if hole_location is None else hole_location
        self.hole_radius = hole_radius
        self.hole_radius_squared = hole_radius * hole_radius
        self.holing_vmax = holing_vmax

    def normal(self, x, y):
        return [0.0, 0.1, np.sqrt(0.99)]

    def impact_function(self, u):
        x, y, vx, vy = u
        x0, y0 = self.hole_location
        squared_distance_to_hole = (x - x0) * (x - x0) + (y - y0) * (y - y0)
        impact = squared_distance_to_hole / self.hole_radius_squared
        v = np.sqrt(vx * vx + vy * vy)
        return v - self.holing_vmax * (1.0 - impact)


In [ ]:
def simple_roll(t, u, green):
    x, y, vx, vy = u
    dx = vx
    dy = vy

    g = GRAVITY
    I_b = MOI_SOLID_SPHERE
    rho_g = green.softness

    nx, ny, nz = green.normal(x, y)
    grade = np.sqrt(1.0 - nz * nz)
    v = np.sqrt(vx * vx + vy * vy)

    dvx = (g / I_b) * (grade * nx - rho_g * vx / v)
    dvy = (g / I_b) * (grade * ny - rho_g * vy / v)
    return [dx, dy, dvx, dvy]


In [ ]:
class MinBallSpeed:
    def __init__(self, min_speed, *, terminal=True, direction=0):
        self.terminal = terminal
        self.direction = direction
        self.min_speed = min_speed

    def __call__(self, t, u, green):
        x, y, vx, vy = u
        v2 = vx * vx + vy * vy
        return v2 - self.min_speed * self.min_speed


In [ ]:
class HoleCapture:
    def __init__(
        self, *, terminal=True, direction=0, vmax=1.63, hole_radius=HOLE_RADIUS
    ):
        self.terminal = terminal
        self.direction = direction
        self.vmax = vmax
        self.hole_radius = hole_radius
        self.hole_radius_squared = hole_radius * hole_radius

    def __call__(self, t, u, green):
        return green.impact_function(u)


In [ ]:
tspan = [0, 10.0]
t_eval = np.linspace(*tspan, 10_001)
x0, y0 = (0.0, 3.0)
vx0, vy0 = (0.0, -3.0)
u0 = np.array([x0, y0, vx0, vy0])

In [ ]:
sol = solve_ivp(
    simple_roll,
    tspan,
    u0,
    method="LSODA",
    events=[MinBallSpeed(1e-6), HoleCapture()],
    args=(Green(12.0),),
)

In [ ]:
sol

In [ ]:
len(sol.t)

In [ ]:
shapely.LineString(sol.y[0:2].T)

In [ ]:
shapely.MultiPoint(sol.y[0:2].T)

In [ ]:
sol.y[0:2].T

In [ ]:
sol.y.T[-1, 0:2]

In [ ]:
hc = HoleCapture()
hc(0, sol.y.T[-1], Green(6.0))

In [ ]:
4.4 * np.cos(3.2), 4.4 * np.sin(3.2)

In [ ]:
np.linspace(*tspan, 1001)

In [ ]:
tspan = [0, 10.0]
t_eval = np.linspace(*tspan, 10_001)
x0, y0 = (3.0, -0.2)

v_init = 3.2
vs = v_init * norm(1, 0.03).ppf(np.linspace(0.025, 0.975, 11))

theta_init = np.pi
thetas = norm(theta_init, np.deg2rad(1.0)).ppf(np.linspace(0.025, 0.975, 11))

end_points = list()
sols = list()
for v, theta in itertools.product(vs, thetas):
    vx0, vy0 = v * np.cos(theta), v * np.sin(theta)
    u0 = np.array([x0, y0, vx0, vy0])
    sol = solve_ivp(
        simple_roll,
        tspan,
        u0,
        method="Radau",
        t_eval=t_eval,
        events=[MinBallSpeed(1e-6), HoleCapture()],
        args=(Green(12.0),),
    )
    end_point = np.hstack(([v, theta], sol.y.T[-1]))
    end_points.append(end_point)
    sols.append(sol)

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["v_end"] = np.sqrt(df["vx_end"]**2 + df["vy_end"]**2)

In [ ]:
end_point_plot = (
    alt.Chart(df)
    .mark_point()
    .encode(
        longitude="x",
        latitude="y",
        color="v_end",
        tooltip=["v_init", "theta", "x", "y", "v_end"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .properties(width=400, height=400)
)


In [ ]:
hole_point = gpd.GeoDataFrame(
    geometry=[shapely.buffer(shapely.Point(0.0, 0.0), HOLE_RADIUS)],
)
hole_point
hole_plot = (
    alt.Chart(hole_point)
    .mark_geoshape(color="black")
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
)


In [ ]:
(hole_plot + end_point_plot)

In [ ]:
norm.ppf(np.linspace(0.025, 0.975, 11))

In [ ]:
df = pd.DataFrame(np.vstack(([sol.t], sol.y)).T, columns=["t", "x", "y", "vx", "vy"])
df["v"] = np.sqrt(df["vx"]**2 + df["vy"]**2)
df

In [ ]:
hole_plot + alt.Chart(df).mark_point(clip=True).encode(
    longitude="x",
    latitude="y",
    color="v",
)

In [ ]:
tspan = [0, 10.0]
t_eval = np.linspace(*tspan, 10_001)
x0, y0 = (0.0, 3.0)
vx0, vy0 = (0.0, -3.0)
u0 = np.array([x0, y0, vx0, vy0])

In [ ]:
sol = solve_ivp(
    simple_roll,
    tspan,
    u0,
    method="LSODA",
    dense_output=True,
    events=[MinBallSpeed(1e-6), HoleCapture()],
    args=(Green(12.0),),
)

In [ ]:
sol

In [ ]:
df = pd.DataFrame(np.vstack(([sol.t], sol.y)).T, columns=["t", "x", "y", "vx", "vy"])
df["v"] = np.sqrt(df["vx"]**2 + df["vy"]**2)
df

In [ ]:
hc = HoleCapture()
hc(sol.t, sol.y.T[-1], Green(12.0))

In [ ]:
df["capture"] = [hc(sol.t[i], sol.y.T[i], Green(12.0)) for i, _ in enumerate(sol.t)]
df["d2"] = df["x"]**2 + df["y"]**2
df["distance"] = np.sqrt(df["d2"])

In [ ]:
hole_plot + alt.Chart(df[df["d2"] < 10.0]).mark_point(clip=True).encode(
    longitude="x",
    latitude="y",
    color=alt.Color("capture", scale=alt.Scale(type="symlog")),
    tooltip=["t", "x", "y", "v", "capture", "distance"]
)

In [ ]:
i_nearest, t_nearest = min(
    enumerate(sol.t), key=lambda x: hc(x[1], sol.y.T[x[0]], Green(12.0))
)
i_nearest, t_nearest

In [ ]:
t_min, t_max = sol.t[[i_nearest - 1, i_nearest]]

In [ ]:
t_eval = np.arange(t_min, t_max+0.01, 0.01)
u_eval = sol.sol(t_eval).T

In [ ]:
captures = np.array([hc(t, u, Green(12.0)) for t, u in zip(t_eval, u_eval)])
captures

In [ ]:
impact_parameter = min(captures)
impact_parameter

In [ ]:
holed = impact_parameter < 0
holed

In [ ]:
np.amin([Green(12.0).impact_function(u) for u in sol.y.T])

In [ ]:
def holing_objective(sol, green):
    i_nearest = np.argmin([green.impact_function(u) for u in sol.y.T])
    try:
        t_min = sol.t[i_nearest - 1]
    except IndexError:
        t_min = sol.t[0]
    try:
        t_max = sol.t[i_nearest + 1]
    except IndexError:
        t_max = sol.t[-1]
    t_eval = np.arange(t_min, t_max + 0.01, 0.01)
    u_eval = sol.sol(t_eval).T
    impact = np.amin([green.impact_function(u) for u in u_eval])
    return impact


In [ ]:
def simulate_rolls(vs, thetas, green, init, *, tspan=(0.0, 10.0), method="LSODA", min_ball_speed_tol=1e-3, **kwargs):
    x0, y0 = init
    end_points = list()
    sols = list()
    impacts = list()
    for v, theta in itertools.product(vs, thetas):
        vx0, vy0 = v * np.cos(theta), v * np.sin(theta)
        u0 = np.array([x0, y0, vx0, vy0])
        sol = solve_ivp(
            simple_roll,
            tspan,
            u0,
            method=method,
            dense_output=True,
            events=[MinBallSpeed(min_ball_speed_tol)],
            args=(green,),
            **kwargs,
        )
        end_point = np.hstack(([v, theta], sol.y.T[-1]))
        impact = holing_objective(sol, green)
        end_points.append(end_point)
        sols.append(sol)
        impacts.append(impact)

    return end_points, sols, impacts

In [96]:
broadie_putts_gained = pd.DataFrame(
    {
        "distance_ft": [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 40, 50, 60, 90],
        "avg_putts": [
            1.0,
            1.01,
            1.05,
            1.14,
            1.24,
            1.34,
            1.43,
            1.5,
            1.56,
            1.61,
            1.78,
            1.87,
            1.98,
            2.06,
            2.14,
            2.21,
            2.36,
        ],
    }
)
ONE_FOOT = 0.305
approx_putts = interp1d(
    broadie_putts_gained["distance_ft"] * ONE_FOOT,
    broadie_putts_gained["avg_putts"],
    fill_value="extrapolate"
)


In [97]:
broadie_putts_gained

,distance_ft,avg_putts
0,0,1.00
1,2,1.01
2,3,1.05
3,4,1.14
4,5,1.24
5,6,1.34
6,7,1.43
7,8,1.50
8,9,1.56
9,10,1.61


In [ ]:
def aim_objective_inner(x, green, init_position, *, v_cv=0.06, theta_sd=np.deg2rad(1.0)):
    v, theta = x
    logging.debug(f"Simulating rolls for bundle centered at {v=}, {theta=}...")
    vs = v * np.geomspace(1.0 - 2 * v_cv, 1.0 + 2 * v_cv, 9)
    thetas = theta + np.linspace(-2 * theta_sd, 2 * theta_sd, 9)
    # vs = v * norm(1, v_cv).ppf(np.linspace(0.025, 0.975, 11))
    # thetas = norm(theta, np.deg2rad(theta_sd)).ppf(np.linspace(0.025, 0.975, 11))
    return simulate_rolls(vs, thetas, green, init_position)


In [ ]:
square_block_2sigma = itertools.product(np.linspace(-2.0, 2.0, 9), np.linspace(-2.0, 2.0, 9))

weights = [
    norm.pdf(x) * norm.pdf(y)
    for x, y in square_block_2sigma
]
weights /= sum(weights)


def aim_objective(*args, **kwargs):
    try:
        end_points, _, impacts = aim_objective_inner(*args, **kwargs)
    except:
        return np.Inf
    scores = [
        approx_putts(np.sqrt(u[2] ** 2 + u[3] ** 2)) if impact >= 0 else 0
        for u, impact in zip(end_points, impacts)
    ]

    return sum(score * weight for score, weight in zip(scores, weights))


In [ ]:
weights

In [ ]:
optim_result = minimize(
    aim_objective,
    (3.0, np.pi),
    args=(Green(12.0), (3.05, 0.0)),
    method="Nelder-Mead",
    bounds=[(0.0, 10.0), (0, 2*np.pi)],
    options={"maxiter": 50, "disp": True},
)
optim_result

In [ ]:
end_points, sols, impacts = aim_objective_inner(optim_result.x, Green(12.0), (3.05, 0.0))

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["impact"] = impacts
df["holed"] = df["impact"] < 0.0
df["weight"] = weights

In [ ]:
sum(df["weight"][df["holed"]])

In [ ]:
hole_plot + (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color="holed",
        size = "weight",
        opacity="weight",
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "weight"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .properties(width=400, height=400)
)

In [ ]:
def take_trajectory(sol):
    df = pd.DataFrame(sol.y.T, columns=["x", "y", "vx", "vy"])
    df["t"] = sol.t
    return(df)
    

In [ ]:
weights_df = pd.DataFrame(weights, columns = ["weight"])
trajectories = pd.concat(
    (take_trajectory(sol).assign(sol_index=i) for (i, sol) in enumerate(sols)),
    ignore_index=True
).merge(weights_df, left_on="sol_index", right_index=True, how="left")

In [ ]:
hole_plot + (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color="holed",
        opacity="weight",
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "weight"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
) + (
    alt.Chart(trajectories)
    .mark_line(color="#888888", strokeWidth=0.5)
    .encode(
        longitude="x",
        latitude="y",
        detail="sol_index:N",
        opacity="weight",
    )
    .properties(width=600, height=300)
)

In [ ]:
weights[1]

In [ ]:
optim_result = minimize(
    aim_objective,
    (2.0, np.pi),
    args=(Green(12.0), (0.915, 0.0)),
    method="Nelder-Mead",
    bounds=[(0.0, 10.0), (0, 2*np.pi)],
    options={"maxiter" :50, "disp": True},
)
optim_result

In [ ]:
end_points, _, impacts = aim_objective_inner(optim_result.x, Green(12.0), (0.915, 0.0))

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["impact"] = impacts
df["holed"] = df["impact"] < 0.0
df["weight"] = weights

In [ ]:
sum(df["weight"][df["holed"]])

In [ ]:
hole_plot + (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color="holed",
        opacity="weight",
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "weight"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .properties(width=400, height=400)
)

In [ ]:
optim_result = minimize(
    aim_objective,
    (2.0, np.pi),
    args=(Green(12.0), (0.61, 0.0)),
    method="Nelder-Mead",
    bounds=[(0.0, 10.0), (0, 2*np.pi)],
    options={"maxiter" :50, "disp": True},
)
optim_result

In [ ]:
end_points, _, impacts = aim_objective_inner(optim_result.x, Green(12.0), (0.61, 0.0))

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["impact"] = impacts
df["holed"] = df["impact"] < 0.0
df["weight"] = weights

In [ ]:
sum(df["weight"][df["holed"]])

In [ ]:
hole_plot + (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color="holed",
        opacity="weight",
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "weight"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .properties(width=400, height=400)
)

In [ ]:
optim_result = minimize(
    aim_objective,
    (4.0, 1.5*np.pi),
    args=(Green(12.0), (0.0, 3.05)),
    method="Nelder-Mead",
    bounds=[(0.0, 10.0), (0, 2*np.pi)],
    options={"maxiter": 50, "disp": True},
)
optim_result

In [ ]:
end_points, _, impacts = aim_objective_inner(optim_result.x, Green(12.0), (0.0, 3.05))

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["impact"] = impacts
df["holed"] = df["impact"] < 0.0
df["weight"] = weights

In [ ]:
sum(df["weight"][df["holed"]])

In [ ]:
hole_plot + (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color="holed",
        opacity="weight",
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "weight"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .properties(width=400, height=400)
)

In [ ]:
optim_result = minimize(
    aim_objective,
    (4.0, 0.5*np.pi),
    args=(Green(12.0), (0.0, -3.05)),
    method="Nelder-Mead",
    bounds=[(0.0, 10.0), (0, 2*np.pi)],
    options={"maxiter": 50, "disp": True},
)
optim_result

In [ ]:
end_points, _, impacts = aim_objective_inner(optim_result.x, Green(12.0), (0.0, -3.05))

In [ ]:
df = pd.DataFrame(end_points, columns=["v_init", "theta", "x", "y", "vx_end", "vy_end"])
df["impact"] = impacts
df["holed"] = df["impact"] < 0.0
df["weight"] = weights

In [ ]:
sum(df["weight"][df["holed"]])

In [ ]:
hole_plot + (
    alt.Chart(df)
    .mark_point(filled=True)
    .encode(
        longitude="x",
        latitude="y",
        color="holed",
        opacity="weight",
        shape="holed",
        tooltip=["v_init", "theta", "x", "y", "impact", "weight"],
    )
    # .project(
    #     type="identity",
    #     reflectY=True,
    # )
    .properties(width=400, height=400)
)

In [ ]:
sum([
    norm.pdf(x) * norm.pdf(y)
    for x, y in itertools.product(
        np.linspace(-2.0, 2.0, 9), np.linspace(-2.0, 2.0, 9)
    )
])

In [ ]:
sum([
    norm.pdf(x) * norm.pdf(y)
    for x, y in itertools.product(
        np.linspace(-2.5, 2.5, 11), np.linspace(-2.5, 2.5, 11)
    )
])

In [ ]:
sum([
    norm.pdf(x) * norm.pdf(y)
    for x, y in itertools.product(
        np.linspace(-3.0, 3.0, 13), np.linspace(-3.0, 3.0, 13)
    )
])


In [ ]:
sum([
    norm.pdf(x) * norm.pdf(y)
    for x, y in itertools.product(
        np.linspace(-4.0, 4.0, 17), np.linspace(-4.0, 4.0, 17)
    )
])

In [ ]:
green = Green(12.0)
green.normal(0, 0)